In [6]:
from pathlib import Path
import csv
import math
import struct

import dpkt
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import DBSCAN, KMeans, SpectralClustering
import hdbscan


## Filtering, subsampling, and clustering

In [27]:
def prepare_points(points, min_distance, max_distance, keep_fraction, seed):
    distances = np.linalg.norm(points, axis=1)
    filtered = points[(distances >= min_distance) & (distances <= max_distance)]
    if len(filtered) and keep_fraction < 1.0:
        keep_count = max(1, int(np.ceil(len(filtered) * keep_fraction)))
        rng = np.random.default_rng(seed)
        indices = np.sort(rng.choice(len(filtered), size=keep_count, replace=False))
        filtered = filtered[indices]
    return filtered


def cluster_points(points, method, cluster_count, dbscan_eps, min_samples, hdbscan_min_cluster_size, spectral_neighbors):
    point_count = len(points)
    if point_count == 0:
        return np.empty(0, dtype=int)
    if method == 'K-means':
        return KMeans(n_clusters=min(cluster_count, point_count), n_init=10, random_state=42).fit_predict(points)
    if method == 'Spectral':
        if point_count < 2:
            return np.zeros(point_count, dtype=int)
        count = min(cluster_count, point_count - 1)
        neighbors = min(spectral_neighbors, point_count - 1)
        return SpectralClustering(n_clusters=count, affinity='nearest_neighbors', n_neighbors=neighbors, assign_labels='kmeans', random_state=42).fit_predict(points)
    if method == 'DBSCAN':
        return DBSCAN(eps=dbscan_eps, min_samples=min_samples).fit_predict(points)
    if method == 'HDBSCAN':
        if point_count < 2:
            return np.full(point_count, -1, dtype=int)
        return hdbscan.HDBSCAN(min_cluster_size=min(hdbscan_min_cluster_size, point_count), min_samples=min_samples).fit_predict(points)
    raise ValueError(f'Unknown clustering method: {method}')


def cluster_frame(points, method, settings, seed):
    selected = prepare_points(points, settings['min_distance'], settings['max_distance'], settings['keep_fraction'], seed)
    labels = cluster_points(selected, method, settings['cluster_count'], settings['dbscan_eps'], settings['min_samples'], settings['hdbscan_min_cluster_size'], settings['spectral_neighbors'])
    cluster_labels = set(labels.tolist()) - {-1}
    return {
        'points': selected,
        'labels': labels,
        'decoded_count': len(points),
        'used_count': len(selected),
        'cluster_count': len(cluster_labels),
        'noise_count': int(np.count_nonzero(labels == -1)),
    }

## Open file

In [22]:
PCAP_PATH = Path("../data/myself in my room dancing.pcap")
if not PCAP_PATH.exists():
    PCAP_PATH = Path("data/driveway.pcap")

CALIBRATION_PATH = Path("../Lidar setup/JT128_angle_correction.csv")
if not CALIBRATION_PATH.exists():
    CALIBRATION_PATH = Path("Lidar setup/JT128_angle_correction.csv")

MAX_FRAMES = 40
if isinstance(MAX_FRAMES, bool) or not isinstance(MAX_FRAMES, int) or MAX_FRAMES < 1:
    raise ValueError("MAX_FRAMES must be a positive integer.")


def load_calibration(path):
    calibration = {}
    with path.open("r", encoding="utf-8", errors="ignore", newline="") as calibration_file:
        for row in csv.DictReader(calibration_file):
            try:
                channel = int(row["Channel"])
                elevation = float(row["Elevation"])
                azimuth = float(row["Azimuth"])
            except (KeyError, TypeError, ValueError):
                continue
            calibration[channel] = (elevation, azimuth)

    if sorted(calibration) != list(range(1, 129)):
        raise ValueError(f"Expected 128 JT128 calibration rows in {path}.")

    elevations = np.array([calibration[channel][0] for channel in range(1, 129)], dtype=float)
    azimuth_offsets = np.array([calibration[channel][1] for channel in range(1, 129)], dtype=float)
    return elevations, azimuth_offsets


VERTICAL_ANGLES_DEG, AZIMUTH_OFFSETS_DEG = load_calibration(CALIBRATION_PATH)


def decode_packet(payload):
    if len(payload) < 12 or payload[:4] != b"\xee\xff\x01\x04":
        return []

    laser_count, block_count, _, distance_unit_mm, _, status = payload[6:12]
    if laser_count != len(VERTICAL_ANGLES_DEG):
        raise ValueError(f"Packet has {laser_count} channels, but the calibration has {len(VERTICAL_ANGLES_DEG)}.")

    unit_size = 3 + bool(status & 0x10) + bool(status & 0x20) + bool(status & 0x40)
    block_size = 2 + laser_count * unit_size
    optional_size = 4  # CRC following the block data
    optional_size += 17 if status & 0x04 else 0  # Function-safety data
    optional_size += 26  # Hesai packet tail
    optional_size += 4 if status & 0x01 else 0  # Sequence number
    optional_size += 22 if status & 0x02 else 0  # IMU data
    optional_size += 4  # Packet CRC
    optional_size += 32 if status & 0x08 else 0  # Cyber-security signature
    expected_size = 12 + block_count * block_size + optional_size
    if len(payload) != expected_size:
        raise ValueError(f"Unexpected Hesai v1.4 payload size {len(payload)}; expected {expected_size}.")

    distance_unit_m = distance_unit_mm / 1000.0
    decoded_blocks = []
    block_offset = 12
    for _ in range(block_count):
        azimuth_deg = int.from_bytes(payload[block_offset:block_offset + 2], "little") / 100.0
        block_offset += 2
        azimuths = np.radians((azimuth_deg + AZIMUTH_OFFSETS_DEG) % 360.0)
        elevations = np.radians(VERTICAL_ANGLES_DEG)
        distances = np.empty(laser_count, dtype=float)

        for channel in range(laser_count):
            record = payload[block_offset:block_offset + unit_size]
            raw_distance = struct.unpack_from("<H", record)[0]
            distances[channel] = raw_distance * distance_unit_m
            block_offset += unit_size

        horizontal = distances * np.cos(elevations)
        points = np.column_stack((
            horizontal * np.sin(azimuths),
            horizontal * np.cos(azimuths),
            distances * np.sin(elevations),
        ))
        points = points[distances > 0]
        decoded_blocks.append((azimuth_deg, points))

    return decoded_blocks


frames = []
frame_blocks = []
previous_azimuth = None

with PCAP_PATH.open("rb") as capture_file:
    reader = dpkt.pcap.Reader(capture_file)
    if reader.datalink() != dpkt.pcap.DLT_EN10MB:
        raise ValueError("Expected an Ethernet .pcap capture.")

    for _, raw_packet in reader:
        try:
            ethernet = dpkt.ethernet.Ethernet(raw_packet)
            network = ethernet.data
            if isinstance(network, dpkt.ip.IP):
                transport = network.data
            elif isinstance(network, dpkt.ip6.IP6):
                transport = network.data
            else:
                continue
            if not isinstance(transport, dpkt.udp.UDP):
                continue
            decoded_blocks = decode_packet(bytes(transport.data))
        except (AttributeError, dpkt.dpkt.NeedData, ValueError):
            continue

        for azimuth, points in decoded_blocks:
            if previous_azimuth is not None and previous_azimuth - azimuth > 180:
                if frame_blocks:
                    frames.append(np.vstack(frame_blocks))
                    frame_blocks = []
                if len(frames) >= MAX_FRAMES:
                    break
            if len(points):
                frame_blocks.append(points)
            previous_azimuth = azimuth

        if len(frames) >= MAX_FRAMES:
            break

if len(frames) < MAX_FRAMES and frame_blocks:
    frames.append(np.vstack(frame_blocks))
if not frames:
    raise ValueError("No valid Hesai protocol v1.4 LiDAR frames found in the capture.")

first_frame = frames[0]
print(f"Loaded {len(frames)} frame(s) (maximum {MAX_FRAMES}) from {PCAP_PATH.resolve()}")
print("Points per frame: " + ", ".join(f"{len(frame):,}" for frame in frames))

Loaded 40 frame(s) (maximum 40) from C:\Users\danag\Documents\LIDAR\data\myself in my room dancing.pcap
Points per frame: 41,628, 212,466, 211,811, 210,972, 210,335, 209,876, 208,753, 207,858, 207,285, 206,456, 206,094, 206,493, 205,931, 205,823, 205,735, 204,971, 201,732, 198,349, 199,270, 198,899, 202,886, 205,352, 205,981, 206,216, 208,889, 212,438, 213,687, 214,075, 213,973, 214,153, 214,363, 214,412, 214,507, 215,006, 214,956, 214,709, 214,756, 214,733, 214,787, 214,808


## Plot frames in sequence

In [26]:
from IPython.display import clear_output, display
from matplotlib.colors import BoundaryNorm, ListedColormap
import ipywidgets as widgets

PLOT_MIN_DISTANCE_M = 0.0
PLOT_MAX_DISTANCE_M = 3.0
PLOT_KEEP_FRACTION = 0.8
PLOT_RANDOM_SEED = 42

method_widget = widgets.Dropdown(
    options=["K-means", "DBSCAN", "HDBSCAN", "Spectral"],
    value="DBSCAN",
    description="Method",
)
frame_count_widget = widgets.IntSlider(
    value=min(MAX_FRAMES, len(frames)),
    min=1,
    max=len(frames),
    step=1,
    description="Frames",
    continuous_update=False,
)
min_distance_widget = widgets.FloatText(value=PLOT_MIN_DISTANCE_M, description="Min range (m)")
max_distance_widget = widgets.FloatText(value=PLOT_MAX_DISTANCE_M, description="Max range (m)")
keep_fraction_widget = widgets.FloatSlider(
    value=PLOT_KEEP_FRACTION,
    min=0.05,
    max=1.0,
    step=0.05,
    description="Point fraction",
    readout_format=".2f",
    continuous_update=False,
)
cluster_count_widget = widgets.IntSlider(
    value=5,
    min=2,
    max=30,
    step=1,
    description="K-means/Spectral clusters",
    continuous_update=False,
)
dbscan_eps_widget = widgets.FloatText(value=0.1, description="DBSCAN eps (m)")
min_samples_widget = widgets.IntSlider(
    value=10,
    min=1,
    max=100,
    step=1,
    description="Min samples",
    continuous_update=False,
)
hdbscan_min_cluster_size_widget = widgets.IntSlider(
    value=10,
    min=2,
    max=100,
    step=1,
    description="HDBSCAN min cluster size",
    continuous_update=False,
)
spectral_neighbors_widget = widgets.IntSlider(
    value=10,
    min=1,
    max=100,
    step=1,
    description="Spectral neighbors",
    continuous_update=False,
)
random_seed_widget = widgets.IntText(value=PLOT_RANDOM_SEED, description="Sample seed")
run_button = widgets.Button(description="Cluster and plot", button_style="primary", icon="play")
plot_output = widgets.Output()

controls = widgets.VBox([
    widgets.HBox([method_widget, frame_count_widget]),
    widgets.HBox([min_distance_widget, max_distance_widget, keep_fraction_widget]),
    widgets.HBox([cluster_count_widget, dbscan_eps_widget, min_samples_widget]),
    widgets.HBox([hdbscan_min_cluster_size_widget, spectral_neighbors_widget, random_seed_widget]),
])


def plot_selected_frames(_):
    run_button.disabled = True
    try:
        with plot_output:
            clear_output(wait=True)
            selected_frames = frames[:frame_count_widget.value]
            for frame_number, frame in enumerate(selected_frames, start=1):
                plot_points = prepare_points(
                    frame,
                    min_distance=min_distance_widget.value,
                    max_distance=max_distance_widget.value,
                    keep_fraction=keep_fraction_widget.value,
                    seed=random_seed_widget.value + frame_number - 1,
                )
                if not len(plot_points):
                    print(f"Skipping frame {frame_number}: no points remain after filtering")
                    continue

                labels = cluster_points(
                    plot_points,
                    method_widget.value,
                    cluster_count_widget.value,
                    dbscan_eps_widget.value,
                    min_samples_widget.value,
                    hdbscan_min_cluster_size_widget.value,
                    spectral_neighbors_widget.value,
                )
                cluster_ids = np.unique(labels)
                cluster_colors = plt.get_cmap("tab20", len(cluster_ids))(np.arange(len(cluster_ids)))
                if -1 in cluster_ids:
                    cluster_colors[0] = (0.5, 0.5, 0.5, 1.0)
                cluster_cmap = ListedColormap(cluster_colors)
                cluster_norm = BoundaryNorm(np.arange(len(cluster_ids) + 1) - 0.5, cluster_cmap.N)
                color_indices = np.searchsorted(cluster_ids, labels)
                cluster_count = np.count_nonzero(cluster_ids != -1)
                noise_count = np.count_nonzero(labels == -1)

                print(
                    f"Frame {frame_number}/{len(selected_frames)}: plotting {len(plot_points):,} of {len(frame):,} points, "
                    f"{cluster_count} clusters, {noise_count:,} noise points"
                )

                figure = plt.figure(figsize=(10, 8))
                axes = figure.add_subplot(111, projection="3d")
                scatter = axes.scatter(
                    plot_points[:, 0],
                    plot_points[:, 1],
                    plot_points[:, 2],
                    c=color_indices,
                    cmap=cluster_cmap,
                    norm=cluster_norm,
                    s=1,
                )
                axes.set_title(f"Hesai JT128 frame {frame_number} ({method_widget.value})")
                axes.set_xlabel("X (m)")
                axes.set_ylabel("Y (m)")
                axes.set_zlabel("Z (m)")
                axes.set_box_aspect(np.maximum(np.ptp(plot_points, axis=0), 1e-6))
                colorbar = figure.colorbar(scatter, ax=axes, ticks=np.arange(len(cluster_ids)), shrink=0.65)
                colorbar.ax.set_yticklabels(["Noise (-1)" if label == -1 else str(label) for label in cluster_ids])
                colorbar.set_label(f"{method_widget.value} cluster")
                plt.tight_layout()
                plt.show()
                plt.close(figure)
    finally:
        run_button.disabled = False


run_button.on_click(plot_selected_frames)
display(controls, run_button, plot_output)


Button(button_style='primary', description='Cluster and plot', icon='play', style=ButtonStyle())

Output()